# Consolidado días 22–25 — SQL avanzado y pandas avanzado

**Dataset:** Superstore — `train.csv` (9.800 transacciones)  
**Bloque mañana:** SQL GROUP BY con HAVING · JOIN con agregación · `.agg()` nombrado · `transform()` · encadenamiento  
**Bloque tarde:** ejecutar `projects/superstore-analysis/notebooks/01_patrones_ventas.ipynb` + Excel

---

In [1]:
import sqlite3
import pandas as pd

# carga el dataset en pandas
df = pd.read_csv('../data/external/train.csv')

# crea una base de datos SQLite en memoria y vuelca el DataFrame en ella
# esto permite ejecutar SQL real sobre los mismos datos que usa pandas
conn = sqlite3.connect(':memory:')
df.to_sql('ventas', conn, index=False, if_exists='replace')

print(f'Dataset cargado: {df.shape[0]:,} filas x {df.shape[1]} columnas')
print('Columnas disponibles:', list(df.columns))

Dataset cargado: 9,800 filas x 18 columnas
Columnas disponibles: ['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales']


---
# Parte 1 — SQL avanzado

Las dos operaciones más comunes en análisis de datos con SQL: filtrar grupos agregados y combinar tablas con agregación.

## 1.1 — GROUP BY: recordatorio

`GROUP BY` agrupa las filas que comparten el mismo valor en una columna y aplica una función agregada a cada grupo.

```sql
SELECT columna_grupo, SUM(ventas)
FROM tabla
GROUP BY columna_grupo;
```

El orden de ejecución lógico en SQL es siempre: `FROM` → `WHERE` → `GROUP BY` → `HAVING` → `SELECT` → `ORDER BY`.  
Entender ese orden explica por qué ciertas cosas van en `WHERE` y otras en `HAVING`.

In [2]:
# GROUP BY básico — total de ventas por categoría
query = '''
SELECT Category, ROUND(SUM(Sales), 2) AS ventas_totales
FROM ventas
GROUP BY Category
ORDER BY ventas_totales DESC
'''
pd.read_sql_query(query, conn)

,Category,ventas_totales
0,Technology,827455.87
1,Furniture,728658.58
2,Office Supplies,705422.33


## 1.2 — GROUP BY con HAVING

### Por qué existe HAVING

`WHERE` se ejecuta **antes** de `GROUP BY` — en ese momento las sumas y promedios todavía no existen.  
Si necesitás filtrar por un valor agregado (`SUM`, `AVG`, `COUNT`), usás `HAVING`, que se ejecuta **después** del agrupamiento.

| Cláusula | Cuándo se ejecuta | Puede usar agregados |
|----------|-------------------|---------------------|
| `WHERE`  | Antes de GROUP BY | No |
| `HAVING` | Después de GROUP BY | Sí |

```sql
SELECT categoria, SUM(ventas) AS total
FROM tabla
-- WHERE filtra filas individuales (antes de agrupar)
WHERE año = 2023
GROUP BY categoria
-- HAVING filtra grupos (después de agrupar)
HAVING SUM(ventas) > 500000;
```

### Cuándo combinarlos

Se pueden usar los dos en la misma consulta:  
`WHERE` reduce el conjunto de filas antes de agrupar (más eficiente).  
`HAVING` filtra los grupos resultantes por una condición sobre el agregado.

In [3]:
# HAVING simple — categorías con ventas totales superiores a $700.000
query = '''
SELECT
    Category,
    ROUND(SUM(Sales), 2) AS ventas_totales,
    COUNT(*)             AS num_ordenes
FROM ventas
GROUP BY Category
HAVING SUM(Sales) > 700000
ORDER BY ventas_totales DESC
'''
pd.read_sql_query(query, conn)

,Category,ventas_totales,num_ordenes
0,Technology,827455.87,1813
1,Furniture,728658.58,2078
2,Office Supplies,705422.33,5909


In [4]:
# WHERE + HAVING combinados
# WHERE primero reduce las filas (solo región West)
# HAVING después filtra los grupos resultantes por umbral de ventas
query = '''
SELECT
    "Sub-Category",
    ROUND(SUM(Sales), 2)  AS ventas_totales,
    ROUND(AVG(Sales), 2)  AS ticket_medio,
    COUNT(*)              AS num_ordenes
FROM ventas
WHERE Region = 'West'
GROUP BY "Sub-Category"
HAVING SUM(Sales) > 50000
ORDER BY ventas_totales DESC
'''
pd.read_sql_query(query, conn)

,Sub-Category,ventas_totales,ticket_medio,num_ordenes
0,Chairs,100023.20,492.73,203
1,Phones,97859.50,358.46,273
2,Tables,81016.23,716.96,113
3,Storage,69256.20,263.33,263
4,Accessories,60632.01,238.71,254
5,Binders,55173.63,119.42,462


In [5]:
# HAVING con múltiples condiciones
# sub-categorías con más de 200 órdenes Y ticket medio superior a $200
query = '''
SELECT
    "Sub-Category",
    COUNT(*)             AS num_ordenes,
    ROUND(AVG(Sales), 2) AS ticket_medio,
    ROUND(SUM(Sales), 2) AS ventas_totales
FROM ventas
GROUP BY "Sub-Category"
HAVING COUNT(*) > 200
   AND AVG(Sales) > 200
ORDER BY ventas_totales DESC
'''
pd.read_sql_query(query, conn)

,Sub-Category,num_ordenes,ticket_medio,ventas_totales
0,Phones,876,374.18,327782.45
1,Chairs,607,531.83,322822.73
2,Storage,832,263.63,219343.39
3,Tables,314,645.89,202810.63
4,Accessories,756,217.18,164186.70
5,Bookcases,226,503.60,113813.20
6,Appliances,459,227.93,104618.40


## 1.3 — JOIN con agregación

### El patrón más común en análisis

Combinar tablas y luego agregar el resultado. El orden importa:  
**1. FROM + JOIN** (une las tablas)  
**2. WHERE** (filtra filas)  
**3. GROUP BY** (agrupa)  
**4. HAVING** (filtra grupos)

```sql
SELECT t1.col_grupo, SUM(t2.valor)
FROM tabla1 t1
INNER JOIN tabla2 t2 ON t1.id = t2.id
GROUP BY t1.col_grupo
HAVING SUM(t2.valor) > umbral;
```

### Variante: agregar antes de hacer JOIN (subquery)

Cuando cada tabla ya tiene muchas filas conviene reducirlas primero con un agregado y luego unir los resúmenes:

```sql
SELECT a.categoria, a.total_a, b.total_b
FROM
    (SELECT categoria, SUM(ventas) AS total_a FROM tabla_a GROUP BY categoria) a
    INNER JOIN
    (SELECT categoria, SUM(costos)  AS total_b FROM tabla_b GROUP BY categoria) b
    ON a.categoria = b.categoria;
```

### Self-join

Una tabla puede unirse consigo misma. Útil para comparar grupos dentro del mismo dataset:  
comparar el total de cada región con el total nacional, o comparar años dentro del mismo segmento.

In [6]:
# JOIN entre dos vistas derivadas de la misma tabla
# vista A: total de ventas por segmento
# vista B: número de órdenes por segmento
# JOIN une los dos resúmenes en una sola tabla
query = '''
SELECT
    a.Segment,
    a.ventas_totales,
    b.num_ordenes,
    ROUND(a.ventas_totales / b.num_ordenes, 2) AS ticket_medio
FROM
    (SELECT Segment, ROUND(SUM(Sales), 2) AS ventas_totales
     FROM ventas GROUP BY Segment) a
    INNER JOIN
    (SELECT Segment, COUNT(*) AS num_ordenes
     FROM ventas GROUP BY Segment) b
    ON a.Segment = b.Segment
ORDER BY ticket_medio DESC
'''
pd.read_sql_query(query, conn)

,Segment,ventas_totales,num_ordenes,ticket_medio
0,Home Office,424982.18,1746,243.40
1,Corporate,688494.07,2953,233.15
2,Consumer,1148060.53,5101,225.07


In [7]:
# self-join: comparar cada región con el total general
# la subquery calcula el total global
# el JOIN lo adjunta a cada fila del resultado por región
query = '''
SELECT
    r.Region,
    ROUND(r.ventas_region, 2)                        AS ventas_region,
    ROUND(t.ventas_global, 2)                        AS ventas_global,
    ROUND(r.ventas_region / t.ventas_global * 100, 1) AS pct_del_total
FROM
    (SELECT Region, SUM(Sales) AS ventas_region
     FROM ventas GROUP BY Region) r
    CROSS JOIN
    (SELECT SUM(Sales) AS ventas_global FROM ventas) t
ORDER BY pct_del_total DESC
'''
pd.read_sql_query(query, conn)

,Region,ventas_region,ventas_global,pct_del_total
0,West,710219.68,2261536.78,31.4
1,East,669518.73,2261536.78,29.6
2,Central,492646.91,2261536.78,21.8
3,South,389151.46,2261536.78,17.2


In [8]:
# JOIN + HAVING combinados
# filtrar solo los segmentos cuyo ticket medio supere $230
query = '''
SELECT
    a.Segment,
    a.ventas_totales,
    b.num_ordenes,
    ROUND(a.ventas_totales / b.num_ordenes, 2) AS ticket_medio
FROM
    (SELECT Segment, ROUND(SUM(Sales), 2) AS ventas_totales
     FROM ventas GROUP BY Segment) a
    INNER JOIN
    (SELECT Segment, COUNT(*) AS num_ordenes
     FROM ventas GROUP BY Segment) b
    ON a.Segment = b.Segment
HAVING ticket_medio > 230
ORDER BY ticket_medio DESC
'''
pd.read_sql_query(query, conn)

DatabaseError: Execution failed on sql '
SELECT
    a.Segment,
    a.ventas_totales,
    b.num_ordenes,
    ROUND(a.ventas_totales / b.num_ordenes, 2) AS ticket_medio
FROM
    (SELECT Segment, ROUND(SUM(Sales), 2) AS ventas_totales
     FROM ventas GROUP BY Segment) a
    INNER JOIN
    (SELECT Segment, COUNT(*) AS num_ordenes
     FROM ventas GROUP BY Segment) b
    ON a.Segment = b.Segment
HAVING ticket_medio > 230
ORDER BY ticket_medio DESC
': HAVING clause on a non-aggregate query

---
# Parte 2 — pandas avanzado

Los tres patrones que convierten un groupby básico en análisis real: agregaciones limpias, métricas por grupo en cada fila, y pipelines encadenados.

## 2.1 — `.agg()` con agregaciones nombradas

### El problema del `.agg()` básico

```python
df.groupby('Category')['Sales'].agg(['sum', 'mean', 'count'])
```

El resultado tiene columnas que se llaman `sum`, `mean`, `count` — nombres genéricos que no describen qué calculan. Si exportás esto a Excel o lo mergeás con otro DataFrame, hay que renombrar todo después.

### La solución: tuplas nombradas

```python
df.groupby('Category')['Sales'].agg(
    ventas_totales = ('sum'),
    ticket_medio   = ('mean'),
    num_ordenes    = ('count'),
)
```

Cada argumento con nombre se convierte en el nombre de la columna resultado. El valor es la función.  
El resultado ya tiene los nombres correctos — no necesitás `.rename()` después.

### Cuando usás múltiples columnas de origen

Cuando el agrupamiento necesita calcular cosas de columnas distintas, la sintaxis cambia ligeramente:  
cada tupla lleva `(columna_origen, función)`.

```python
df.groupby('Category').agg(
    ventas_totales = ('Sales',    'sum'),
    ticket_medio   = ('Sales',    'mean'),
    num_ordenes    = ('Order ID', 'count'),
    max_descuento  = ('Discount', 'max'),
)
```

In [9]:
# forma básica — columnas con nombres genéricos (evitar en producción)
basico = (
    df.groupby('Category')['Sales']
    .agg(['sum', 'mean', 'count'])
    .reset_index()
)
print('Columnas:', list(basico.columns))
print(basico)

Columnas: ['Category', 'sum', 'mean', 'count']
          Category          sum        mean  count
0        Furniture  728658.5757  350.653790   2078
1  Office Supplies  705422.3340  119.381001   5909
2       Technology  827455.8730  456.401474   1813


In [10]:
# forma nombrada — una sola columna origen
nombrado = (
    df.groupby('Category')['Sales']
    .agg(
        ventas_totales = 'sum',
        ticket_medio   = 'mean',
        num_ordenes    = 'count',
        venta_maxima   = 'max',
    )
    .reset_index()
    .sort_values('ventas_totales', ascending=False)
)
print('Columnas:', list(nombrado.columns))
nombrado.style.format({
    'ventas_totales': '${:,.0f}',
    'ticket_medio'  : '${:,.2f}',
    'venta_maxima'  : '${:,.2f}',
})

Columnas: ['Category', 'ventas_totales', 'ticket_medio', 'num_ordenes', 'venta_maxima']


,Category,ventas_totales,ticket_medio,num_ordenes,venta_maxima
2,Technology,"$827,456",$456.40,1813,"$22,638.48"
0,Furniture,"$728,659",$350.65,2078,"$4,416.17"
1,Office Supplies,"$705,422",$119.38,5909,"$9,892.74"


In [11]:
# forma completa — múltiples columnas de origen
# cada métrica especifica (columna_origen, función) como tupla
completo = (
    df.groupby(['Region', 'Category'])
    .agg(
        ventas_totales = ('Sales',     'sum'),
        ticket_medio   = ('Sales',     'mean'),
        num_ordenes    = ('Order ID',  'count'),
        max_descuento  = ('Discount',  'max'),
    )
    .reset_index()
    .sort_values(['Region', 'ventas_totales'], ascending=[True, False])
)
completo.style.format({
    'ventas_totales': '${:,.0f}',
    'ticket_medio'  : '${:,.2f}',
    'max_descuento' : '{:.0%}',
})

KeyError: "Label(s) ['Discount'] do not exist"

## 2.2 — `transform()`: métricas de grupo sin reducir filas

### El problema que resuelve

`.agg()` reduce el DataFrame — de 9.800 filas pasa a 3 (una por categoría).  
Si después necesitás calcular el porcentaje de cada fila sobre su categoría, tendrías que hacer un `merge()` para volver a unir el total de la categoría con cada fila.

`transform()` lo hace en un solo paso: calcula el agregado de cada grupo y lo devuelve en **el mismo shape** que el DataFrame original.

| | `.agg()` | `transform()` |
|-|----------|---------------|
| Filas resultado | Una por grupo | Mismas que el original |
| Uso típico | Crear tabla resumen | Añadir columna de grupo a cada fila |
| Encadena con merge | No (ya es el resumen) | No necesita merge |

### Cómo funciona

```python
df['total_categoria'] = df.groupby('Category')['Sales'].transform('sum')
```

Para cada fila, busca a qué categoría pertenece, calcula la suma de esa categoría y escribe ese valor en la fila.  
Todas las filas de `Furniture` tendrán el mismo valor en `total_categoria`: el total de Furniture.

### Para qué sirve

- Calcular porcentajes de cada fila sobre el total de su grupo
- Filtrar filas que pertenecen a grupos que cumplen una condición (equivalente al HAVING de SQL pero a nivel de fila)
- Añadir rankings dentro de grupos

In [12]:
# transform('sum') — añade el total de cada categoría a cada fila
# el DataFrame sigue teniendo 9.800 filas — solo se añade una columna nueva
df_work = df[['Category', 'Sales']].copy()

df_work['total_categoria'] = df_work.groupby('Category')['Sales'].transform('sum')

# verificación: todos los registros de Furniture deben tener el mismo total_categoria
print('Shape original:', df_work.shape)
print()
print('Total por categoría (debe ser el mismo en todas las filas del grupo):')
print(df_work.groupby('Category')['total_categoria'].first())

Shape original: (9800, 3)

Total por categoría (debe ser el mismo en todas las filas del grupo):
Category
Furniture          728658.5757
Office Supplies    705422.3340
Technology         827455.8730
Name: total_categoria, dtype: float64


In [13]:
# caso de uso principal: calcular porcentaje de cada venta sobre el total de su categoría
df_pct = df[['Category', 'Sub-Category', 'Sales']].copy()

df_pct['total_categoria'] = df_pct.groupby('Category')['Sales'].transform('sum')
df_pct['pct_sobre_categoria'] = (df_pct['Sales'] / df_pct['total_categoria'] * 100).round(2)

# mostrar algunas filas para verificar que el porcentaje tiene sentido
print(df_pct[['Category', 'Sales', 'total_categoria', 'pct_sobre_categoria']].head(10))

          Category     Sales  total_categoria  pct_sobre_categoria
0        Furniture  261.9600      728658.5757                 0.04
1        Furniture  731.9400      728658.5757                 0.10
2  Office Supplies   14.6200      705422.3340                 0.00
3        Furniture  957.5775      728658.5757                 0.13
4  Office Supplies   22.3680      705422.3340                 0.00
5        Furniture   48.8600      728658.5757                 0.01
6  Office Supplies    7.2800      705422.3340                 0.00
7       Technology  907.1520      827455.8730                 0.11
8  Office Supplies   18.5040      705422.3340                 0.00
9  Office Supplies  114.9000      705422.3340                 0.02


In [14]:
# equivalente al HAVING de SQL: filtrar filas cuyos grupos cumplen una condición
# SQL: SELECT ... FROM ventas GROUP BY Category HAVING SUM(Sales) > 750000
# pandas con transform: se puede hacer sin reducir el DataFrame

df_filter = df.copy()
df_filter['total_categoria'] = df_filter.groupby('Category')['Sales'].transform('sum')

# filtrar solo las filas que pertenecen a categorías con ventas totales > 750.000
categorias_grandes = df_filter[df_filter['total_categoria'] > 750_000]

print('Filas en categorías con ventas > $750.000:', len(categorias_grandes))
print('Categorías incluidas:', categorias_grandes['Category'].unique())

Filas en categorías con ventas > $750.000: 1813
Categorías incluidas: <StringArray>
['Technology']
Length: 1, dtype: str


In [15]:
# comparación agg() vs transform() — para entender la diferencia visual

# agg() reduce → 3 filas
resumen_agg = df.groupby('Category')['Sales'].agg(total='sum').reset_index()
print(f'agg() → {len(resumen_agg)} filas')
print(resumen_agg)
print()

# transform() no reduce → 9.800 filas, columna nueva con el total del grupo
df_copy = df[['Category', 'Sales']].copy()
df_copy['total_categoria'] = df_copy.groupby('Category')['Sales'].transform('sum')
print(f'transform() → {len(df_copy)} filas')
print(df_copy.head(6))

agg() → 3 filas
          Category        total
0        Furniture  728658.5757
1  Office Supplies  705422.3340
2       Technology  827455.8730

transform() → 9800 filas
          Category     Sales  total_categoria
0        Furniture  261.9600      728658.5757
1        Furniture  731.9400      728658.5757
2  Office Supplies   14.6200      705422.3340
3        Furniture  957.5775      728658.5757
4  Office Supplies   22.3680      705422.3340
5        Furniture   48.8600      728658.5757


## 2.3 — Encadenamiento de operaciones

### Por qué encadenar

Sin encadenamiento, cada transformación necesita una variable intermedia:

```python
paso1 = df.groupby('Category')['Sales'].sum().reset_index()
paso2 = paso1.sort_values('Sales', ascending=False)
paso3 = paso2[paso2['Sales'] > 700000]
paso4 = paso3.rename(columns={'Sales': 'ventas_totales'})
```

Con encadenamiento, todo en una expresión dentro de paréntesis:

```python
resultado = (
    df.groupby('Category')['Sales']
    .sum()
    .reset_index()
    .sort_values('Sales', ascending=False)
    .query('Sales > 700000')
    .rename(columns={'Sales': 'ventas_totales'})
)
```

### Reglas

1. Los paréntesis externos son lo que permite el salto de línea — sin ellos Python no entiende que la expresión continúa
2. Cada método devuelve el objeto transformado para que el siguiente método lo reciba
3. No todos los métodos son encadenables — los que devuelven `None` (como `.sort_values(inplace=True)`) rompen la cadena. Usar siempre la versión sin `inplace=`

### Métodos más usados en la cadena

| Método | Qué hace en la cadena |
|--------|-----------------------|
| `.groupby().agg()` | Agrega por grupo |
| `.reset_index()` | Convierte índice en columna |
| `.sort_values()` | Ordena |
| `.query('condicion')` | Filtra filas con expresión string |
| `.rename(columns={})` | Renombra columnas |
| `.assign(col=expr)` | Añade columna calculada |
| `.head(n)` | Limita a N filas |
| `.reset_index(drop=True)` | Reinicia índice después de filtrar |

In [ ]:
# pipeline básico: agrupar → ordenar → tomar top N
top5_subcategorias = (
    df.groupby('Sub-Category')['Sales']
    .sum()
    .reset_index()
    .sort_values('Sales', ascending=False)
    .head(5)
    .reset_index(drop=True)   # reiniciar índice para que empiece en 0
    .rename(columns={'Sales': 'ventas_totales'})
)
top5_subcategorias

In [ ]:
# pipeline con .assign() para añadir columnas calculadas dentro de la cadena
# .assign() recibe el nombre de la columna nueva y una función lambda que recibe el df actual
total_global = df['Sales'].sum()

resumen_region = (
    df.groupby('Region')['Sales']
    .sum()
    .reset_index()
    .sort_values('Sales', ascending=False)
    .rename(columns={'Sales': 'ventas_totales'})
    .assign(
        pct_del_total = lambda x: (x['ventas_totales'] / total_global * 100).round(1)
    )
    .reset_index(drop=True)
)
resumen_region.style.format({'ventas_totales': '${:,.0f}', 'pct_del_total': '{:.1f}%'})

In [ ]:
# pipeline completo: agrupar por dos columnas → filtrar grupos → añadir porcentaje → ordenar
# equivale a la consulta SQL del bloque anterior (JOIN + HAVING + cálculo de porcentaje)

total_global = df['Sales'].sum()

analisis_completo = (
    df.groupby(['Region', 'Category'])
    .agg(
        ventas_totales = ('Sales',    'sum'),
        ticket_medio   = ('Sales',    'mean'),
        num_ordenes    = ('Order ID', 'count'),
    )
    .reset_index()
    .assign(
        pct_global = lambda x: (x['ventas_totales'] / total_global * 100).round(2)
    )
    .query('ventas_totales > 100_000')   # equivalente al HAVING SQL
    .sort_values(['Region', 'ventas_totales'], ascending=[True, False])
    .reset_index(drop=True)
)

analisis_completo.style.format({
    'ventas_totales': '${:,.0f}',
    'ticket_medio'  : '${:,.2f}',
    'pct_global'    : '{:.2f}%',
})

---
## Bloque tarde — instrucciones

**1. Ejecutar el proyecto de patrones:**  
Abrí `projects/superstore-analysis/notebooks/01_patrones_ventas.ipynb` y ejecutá celda por celda.  
No leas el código — leé los **resultados**. En cada sección anotá un número que te llame la atención.

**2. Armar el dashboard en Excel:**  
Cargá `projects/superstore-analysis/datos/crudos/train.csv`.

| # | Tabla dinámica | Filas | Columnas | Valores |
|---|----------------|-------|----------|---------|
| 1 | Revenue por categoría | `Category` | — | `Sales` (suma) |
| 2 | Evolución temporal | `Order Date` (agrupar por mes+año) | `Category` | `Sales` (suma) |

Desde la tabla 1: gráfico de **barras horizontales**.  
Desde la tabla 2: gráfico de **líneas** (tiempo en eje X, una línea por categoría).  
Ambos en una hoja `Dashboard`, sin superposición.